# Шикарный видео курс на русском по работе с Yolo
https://www.youtube.com/watch?v=GGmnkEsgf50&list=PLou0mfoS85QU-U4DKZ8kPbuHjy4hsMQYl

# Классификация

1. Загрузка модели

In [ ]:
model = YOLO('yolov8n.pt')

yolov8n.pt - нано-версия (самая быстрая)

Доступны варианты: n, s, m, l, x (от самого быстрого к самому точному)

2. Обработка кадра

In [ ]:
results = model(frame)

Модель возвращает объект с:

Bounding boxes (координаты объектов)

Class IDs (идентификаторы классов)

Confidence scores (уверенность)

3. Визуализация результатов

In [ ]:
annotated_frame = results[0].plot()

Автоматически рисует:

Прямоугольники вокруг объектов

Подписи с названиями классов

Проценты уверенности

4. Классы объектов
YOLOv8 распознает 80 классов из датасета COCO:

🚗 Транспорт: car, bus, truck

🚦 Дорожные объекты: traffic light, stop sign

👥 Люди: person

🐕 Животные: dog, cat, bird

🏠 Предметы: chair, book, laptop

Распознавание объектов на видео

In [ ]:
import cv2
from ultralytics import YOLO

# Загрузка модели YOLO
model = YOLO('yolov8n.pt')

# Загрузка видео файла
video_path = 'Prim.mp4'  # Укажите путь к вашему видео
cap = cv2.VideoCapture(video_path)

while True:
    # Чтение кадра
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось или не удалось прочитать кадр")
        break
    # Детекция объектов с YOLO
    results = model(frame)     
    # Проверяем, есть ли результаты
    
    if len(results) > 0:
        result = results[0]
        # Обрабатываем каждый обнаруженный объект
        for box in result.boxes:
            class_id = int(box.cls[0])
            confidence = float(box.conf[0])
        
        annotated_frame = results[0].plot()

    # Отображение результата
    cv2.imshow('YOLO Video Detection', annotated_frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Распознавание объектов на видео с камеры

In [ ]:
import cv2
import urllib.request
import numpy as np
from ultralytics import YOLO

# Загрузка модели YOLO
model = YOLO('yolov8n.pt')

# Создание объекта для захвата видео
cap = cv2.VideoCapture(0)

while True:
    # Чтение кадра
    ret, frame = cap.read()
    if not ret:
        print("Не удалось получить кадр")
        break

    # Детекция объектов с YOLO
    results = model(frame)
    annotated_frame = results[0].plot()

    # Отображение результата
    cv2.imshow('YOLO Camera', annotated_frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Сегментация

Основные отличия от детекции:

yolov8n-seg.pt вместо yolov8n.pt

Модель выдает маски (точные контуры) вместо прямоугольников

result.plot() автоматически рисует цветные маски

Более точное определение границ объектов

Результат: Объекты будут выделены цветными масками с точными контурами вместо прямоугольников!

Распознавание объектов на видео

In [ ]:
import cv2
from ultralytics import YOLO

# Загрузка модели YOLO для сегментации
model = YOLO('yolov8n-seg.pt')  # seg - segmentation

# Загрузка видео файла
video_path = 'prim2.mp4'
cap = cv2.VideoCapture(video_path)

while True:
    # Чтение кадра
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось")
        break
    
    # Сегментация объектов с YOLO
    results = model(frame)

    # Проверяем, есть ли результаты
    if len(results) > 0:
        result = results[0]
        for i, mask in enumerate(result.masks.data):
            class_id = int(result.boxes.cls[i])
            confidence = float(result.boxes.conf[i])
        # Автоматическая отрисовка масок и bounding boxes
        annotated_frame = results[0].plot()
        
    else:
        annotated_frame = frame

    # Отображение результата
    cv2.imshow('YOLO Segmentation', annotated_frame)

    # Выход по нажатию 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Распознавание объектов на видео с камеры

Подготовка смартфона:

Установите IP Webcam (Android)

Запустите приложение и запишите IP-адрес

В коде замените 192.168.1.100 на ваш IP

Убедитесь, что телефон и компьютер в одной сети Wi-Fi

In [ ]:
import cv2
from ultralytics import YOLO
import time

# Загрузка модели
model = YOLO('yolov8n-seg.pt')

# URL видеопотока с телефона
url = 'http://192.168.1.30:8080/video'  # Замените на ваш IP

# Создание объекта для захвата видео
cap = cv2.VideoCapture(url)

while True:
    ret, frame = cap.read()
    if not ret:
        print("Потеряно соединение с телефоном")
        break
    
    frame_count += 1
    
    # Сегментация каждого 3-го кадра для производительности
    if frame_count % 3 == 0:
        results = model(frame)
        if len(results) > 0:
            annotated_frame = results[0].plot()
            
            # FPS calculation
            fps = frame_count / (time.time() - start_time)
            cv2.putText(annotated_frame, f"FPS: {fps:.1f}", 
                       (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        else:
            annotated_frame = frame
    else:
        annotated_frame = frame

    cv2.imshow('YOLO Segmentation - Phone Camera', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Распознавание объектов на видео, используя supervison

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv

model = YOLO('yolov8n-seg.pt')
mask_annotator = sv.MaskAnnotator()
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

video_path = 'prim2.mp4'
cap = cv2.VideoCapture(video_path)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    results = model(frame)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # Аннотации
    annotated_frame = mask_annotator.annotate(frame.copy(), detections)
    annotated_frame = box_annotator.annotate(annotated_frame, detections)
    annotated_frame = label_annotator.annotate(annotated_frame, detections)
    
    cv2.imshow('YOLO Segmentation', annotated_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Трекер

Инициализация трекера
ByteTrack - современный алгоритм трекинга

In [ ]:
tracker = sv.ByteTrack()

Трекинг объектов - присваиваем ID
Обновляем детекции с информацией о трекинге

In [ ]:
vehicle_detections = tracker.update_with_detections(vehicle_detections)

Создаем подписи с ID трекера

In [ ]:
labels = [
    f"{class_names[class_id]} {tracker_id}"
    for class_id, tracker_id in zip(vehicle_detections.class_id, vehicle_detections.tracker_id)
]

Каждый автомобиль и автобус получает уникальный ID, который сохраняется при движении объекта по кадрам!

In [ ]:
import cv2
from ultralytics import YOLO
import supervision as sv
import numpy as np

# Загрузка модели YOLO для сегментации
model = YOLO('yolov8n-seg.pt')

# Инициализация трекера
tracker = sv.ByteTrack()

# Загрузка видео файла
video_path = 'prim3.mp4'
cap = cv2.VideoCapture(video_path)

# Создаем аннотаторы для сегментации
mask_annotator = sv.MaskAnnotator()
label_annotator = sv.LabelAnnotator()
box_annotator = sv.BoxAnnotator()

# Классы для детекции: car=2, bus=5
target_classes = [2, 5]
class_names = {2: 'CAR', 5: 'BUS'}
colors = {2: (0,255,0),  # Зеленый для автомобилей
          5: (255,0,0)}  # Красный для автобусов

while True:
    ret, frame = cap.read()
    if not ret:
        print("Видео закончилось")
        break
    
    frame = cv2.resize(frame, (640, 480))
    
    # Сегментация объектов с YOLO
    results = model(frame)
    
    # Конвертируем результаты в формат supervision
    detections = sv.Detections.from_ultralytics(results[0])
    
    # Фильтруем только автомобили и автобусы
    vehicle_indices = [
        i for i, class_id in enumerate(detections.class_id) 
        if class_id in target_classes
    ]
    
    vehicle_detections = detections[vehicle_indices]
    
    # Трекинг объектов
    vehicle_detections = tracker.update_with_detections(vehicle_detections)
    
    # Создаем подписи с ID
    labels = [
        f"{class_names[class_id]} {tracker_id}"
        for class_id, tracker_id in zip(vehicle_detections.class_id, vehicle_detections.tracker_id)
    ]
    
    # Аннотируем
    annotated_frame = mask_annotator.annotate(frame.copy(), vehicle_detections)
    annotated_frame = box_annotator.annotate(annotated_frame, vehicle_detections)
    annotated_frame = label_annotator.annotate(annotated_frame, vehicle_detections, labels=labels)
    
    # Отображение результата
    cv2.imshow('Vehicle Segmentation with Tracking', annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()